
# Descriptive Statistics and Visualization Notebook

This notebook focuses on **descriptive statistics** and **visualizations** for minute-level Chinese futures contracts. It is designed to work with multiple contracts in one run while saving figures and tables into contract-specific folders. All titles use a short contract tag (first two Latin letters) to avoid missing Chinese fonts.


In [ ]:

import os
from pathlib import Path
from typing import List, Optional, Tuple, Dict
from datetime import datetime
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf

plt.style.use('seaborn-v0_8-whitegrid')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


In [ ]:

# -----------------------------
# Configuration knobs
# -----------------------------
DATA_DIR = Path('2005年__20250905')
MAX_FILES: Optional[int] = 2
NROWS_PER_FILE: Optional[int] = None
START_DATE: Optional[str] = '2005-01-01'
END_DATE: Optional[str] = '2013-12-31'
RESAMPLE_RULE: str = '5T'
RANDOM_SEED: int = 42
ACF_LAGS: int = 20
SAMPLE_POINTS: int = 10000

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = Path('outputs') / 'descriptive_stats' / RUN_TIMESTAMP
FIG_ROOT = OUTPUT_ROOT / 'figures'
METRIC_ROOT = OUTPUT_ROOT / 'metrics'
for path in [FIG_ROOT, METRIC_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Run timestamp: {RUN_TIMESTAMP}")
print(f"Figures root: {FIG_ROOT.resolve()}")
print(f"Metrics root: {METRIC_ROOT.resolve()}")


In [ ]:

# -----------------------------
# Helper functions
# -----------------------------

def list_csv_files(data_dir: Path, limit: Optional[int] = None) -> List[Path]:
    files = sorted(data_dir.glob('*.csv'))
    if limit is not None:
        files = files[:limit]
    print(f"Found {len(files)} CSV files (showing up to 5):")
    for name in files[:5]:
        print(f"  - {name.name}")
    return files


def contract_short_tag(path: Path) -> str:
    """Extract a two-letter ASCII tag from the contract file name."""
    letters = re.findall(r'[A-Za-z]', path.stem)
    if len(letters) >= 2:
        return ''.join(letters[:2]).upper()
    return path.stem[:2].upper()


def make_contract_dirs(short_tag: str) -> Tuple[Path, Path]:
    fig_dir = FIG_ROOT / short_tag
    metric_dir = METRIC_ROOT / short_tag
    fig_dir.mkdir(parents=True, exist_ok=True)
    metric_dir.mkdir(parents=True, exist_ok=True)
    return fig_dir, metric_dir


def save_table(df: pd.DataFrame, name: str, metric_dir: Path) -> Path:
    path = metric_dir / f"{name}_{RUN_TIMESTAMP}.csv"
    df.to_csv(path, index=False)
    print(f"Saved table to {path}")
    return path


def save_figure(fig: plt.Figure, name: str, fig_dir: Path, short_tag: str) -> Path:
    path = fig_dir / f"{name}_{short_tag}_{RUN_TIMESTAMP}.png"
    fig.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved figure to {path}")
    return path


def load_single_csv(path: Path, start_date: Optional[str], end_date: Optional[str], nrows: Optional[int]) -> pd.DataFrame:
    print(f"[Loading] {path.name}")
    df = pd.read_csv(path, parse_dates=['index'], nrows=nrows)
    df = df.sort_values('index').reset_index(drop=True)
    if start_date is not None:
        df = df[df['index'] >= pd.to_datetime(start_date)]
    if end_date is not None:
        df = df[df['index'] <= pd.to_datetime(end_date)]
    df = df.rename(columns={'index': 'timestamp'})
    df = df.set_index('timestamp')
    print(f"[Loaded] shape after filtering: {df.shape}")
    return df


def add_return_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['r'] = 100 * np.log(df['close']).diff()
    df['abs_r'] = df['r'].abs()
    df['r2'] = df['r'] ** 2
    df['range'] = df['high'] - df['low']
    df['log_volume'] = np.log(df['volume'].replace(0, np.nan))
    df['time_of_day'] = df.index.strftime('%H:%M')
    return df


def basic_describe(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    desc = df[cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).reset_index().rename(columns={'index': 'stat'})
    return desc


def return_stats(series: pd.Series) -> pd.DataFrame:
    stats_dict = {
        'mean': series.mean(),
        'std': series.std(),
        'skew': stats.skew(series.dropna()),
        'kurtosis': stats.kurtosis(series.dropna(), fisher=False),
        'p01': series.quantile(0.01),
        'p05': series.quantile(0.05),
        'p50': series.quantile(0.50),
        'p95': series.quantile(0.95),
        'p99': series.quantile(0.99),
    }
    return pd.DataFrame([stats_dict])


In [ ]:

# -----------------------------
# Plotting helpers
# -----------------------------

def plot_close_volume(df: pd.DataFrame, fig_dir: Path, short_tag: str, resample_rule: str = '5T'):
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    close_resampled = df['close'].resample(resample_rule).last()
    volume_resampled = df['volume'].resample(resample_rule).sum()
    axes[0].plot(close_resampled.index, close_resampled.values, color='steelblue')
    axes[0].set_ylabel('Close price')
    axes[0].set_title(f"{short_tag} - Close price over time")
    axes[1].plot(volume_resampled.index, volume_resampled.values, color='darkorange')
    axes[1].set_ylabel('Volume')
    axes[1].set_title(f"{short_tag} - Trading volume over time")
    axes[1].tick_params(axis='x', rotation=20)
    save_figure(fig, 'close_volume_ts', fig_dir, short_tag)


def plot_price_volume_scatter(df: pd.DataFrame, fig_dir: Path, short_tag: str, sample_points: int):
    data = df.dropna(subset=['close', 'volume'])
    if len(data) > sample_points:
        data = data.sample(sample_points, random_state=RANDOM_SEED)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=data, x='close', y='volume', ax=ax, alpha=0.6, edgecolor=None)
    ax.set_title(f"{short_tag} - Close price vs volume")
    ax.set_xlabel('Close price')
    ax.set_ylabel('Volume')
    save_figure(fig, 'price_vs_volume_scatter', fig_dir, short_tag)


def plot_return_timeseries(df: pd.DataFrame, fig_dir: Path, short_tag: str, max_points: int = 5000):
    series = df['r'].dropna()
    if len(series) > max_points:
        series = series.iloc[:max_points]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(series.index, series.values, color='slategray')
    ax.set_title(f"{short_tag} - Minute returns (first {len(series)} points)")
    ax.set_ylabel('Return (bp)')
    ax.set_xlabel('Time')
    ax.tick_params(axis='x', rotation=20)
    save_figure(fig, 'returns_timeseries', fig_dir, short_tag)


def plot_return_histograms(df: pd.DataFrame, fig_dir: Path, short_tag: str):
    r = df['r'].dropna()
    if r.empty:
        print(f"{short_tag}: no returns to plot.")
        return
    mu, sigma = r.mean(), r.std()
    x_vals = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
    norm_pdf = stats.norm.pdf(x_vals, mu, sigma)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(r, bins=50, stat='density', color='skyblue', ax=axes[0])
    axes[0].plot(x_vals, norm_pdf, color='darkred', label='Gaussian PDF')
    axes[0].set_title(f"{short_tag} - Return distribution vs Gaussian")
    axes[0].legend()

    sns.histplot(r, bins=50, stat='density', color='skyblue', ax=axes[1])
    axes[1].plot(x_vals, norm_pdf, color='darkred', label='Gaussian PDF')
    axes[1].set_yscale('log')
    axes[1].set_title(f"{short_tag} - Return distribution (log scale)")
    axes[1].legend()

    save_figure(fig, 'return_histograms', fig_dir, short_tag)


def plot_volatility_proxy(df: pd.DataFrame, fig_dir: Path, short_tag: str, window_points: int = 5000):
    series = df['abs_r'].dropna()
    if len(series) > window_points:
        series = series.iloc[:window_points]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(series.index, series.values, color='purple')
    ax.set_title(f"{short_tag} - Absolute returns (first {len(series)} points)")
    ax.set_ylabel('|r|')
    ax.set_xlabel('Time')
    ax.tick_params(axis='x', rotation=20)
    save_figure(fig, 'abs_return_series', fig_dir, short_tag)


def plot_acf_panels(df: pd.DataFrame, fig_dir: Path, short_tag: str, lags: int = 20):
    r = df['r'].dropna()
    abs_r = df['abs_r'].dropna()
    r2 = df['r2'].dropna()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    plot_acf(r, lags=lags, ax=axes[0])
    axes[0].set_title(f"{short_tag} - ACF of r")
    plot_acf(abs_r, lags=lags, ax=axes[1])
    axes[1].set_title(f"{short_tag} - ACF of |r|")
    plot_acf(r2, lags=lags, ax=axes[2])
    axes[2].set_title(f"{short_tag} - ACF of r^2")
    fig.tight_layout()
    save_figure(fig, 'acf_panels', fig_dir, short_tag)


def plot_intraday_patterns(df: pd.DataFrame, fig_dir: Path, short_tag: str):
    grouped_abs = df.groupby('time_of_day')['abs_r'].mean().dropna()
    grouped_vol = df.groupby('time_of_day')['volume'].mean().dropna()
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=False)
    axes[0].plot(grouped_abs.index, grouped_abs.values, color='teal')
    axes[0].set_title(f"{short_tag} - Intraday mean |r|")
    axes[0].set_xlabel('Time of day')
    axes[0].set_ylabel('Mean |r|')
    axes[0].tick_params(axis='x', rotation=90)

    axes[1].plot(grouped_vol.index, grouped_vol.values, color='orange')
    axes[1].set_title(f"{short_tag} - Intraday mean volume")
    axes[1].set_xlabel('Time of day')
    axes[1].set_ylabel('Mean volume')
    axes[1].tick_params(axis='x', rotation=90)

    fig.tight_layout()
    save_figure(fig, 'intraday_patterns', fig_dir, short_tag)


def plot_volatility_vs_volume(df: pd.DataFrame, fig_dir: Path, short_tag: str, sample_points: int):
    data = df.dropna(subset=['abs_r', 'log_volume'])
    if len(data) > sample_points:
        data = data.sample(sample_points, random_state=RANDOM_SEED)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=data, x='log_volume', y='abs_r', ax=ax, alpha=0.6, edgecolor=None)
    ax.set_title(f"{short_tag} - |r| vs log(volume)")
    ax.set_xlabel('log(volume)')
    ax.set_ylabel('|r|')
    save_figure(fig, 'absr_vs_logvol', fig_dir, short_tag)


def plot_correlation_heatmap(df: pd.DataFrame, fig_dir: Path, short_tag: str):
    cols = ['r', 'abs_r', 'r2', 'volume', 'log_volume', 'range', 'money']
    available = [c for c in cols if c in df.columns]
    corr = df[available].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
    ax.set_title(f"{short_tag} - Correlation heatmap")
    save_figure(fig, 'correlation_heatmap', fig_dir, short_tag)


In [ ]:

# -----------------------------
# Multi-contract comparison helper
# -----------------------------

def multi_contract_volatility_summary(contract_results: Dict[str, Dict], metric_dir: Path, fig_dir: Path):
    summary_rows = []
    for tag, res in contract_results.items():
        r = res['data']['r'].dropna()
        if r.empty:
            continue
        summary_rows.append({
            'contract': tag,
            'return_std': r.std(),
            'abs_return_mean': res['data']['abs_r'].dropna().mean(),
            'range_mean': res['data']['range'].dropna().mean(),
        })
    if not summary_rows:
        print('No data for multi-contract summary.')
        return
    summary_df = pd.DataFrame(summary_rows)
    save_table(summary_df, 'multi_contract_volatility', metric_dir)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=summary_df, x='contract', y='return_std', ax=ax, palette='Blues_d')
    ax.set_title('Return volatility by contract')
    ax.set_xlabel('Contract (short tag)')
    ax.set_ylabel('Std of returns')
    save_figure(fig, 'multi_contract_return_std', fig_dir, 'ALL')


In [ ]:

# -----------------------------
# Main pipeline per contract
# -----------------------------

def run_descriptive_pipeline(csv_path: Path) -> Dict:
    short_tag = contract_short_tag(csv_path)
    fig_dir, metric_dir = make_contract_dirs(short_tag)

    df_raw = load_single_csv(csv_path, START_DATE, END_DATE, NROWS_PER_FILE)
    if df_raw.empty:
        print(f"{short_tag}: no data after filtering, skipping.")
        return {}

    info_table = pd.DataFrame({
        'rows': [len(df_raw)],
        'start_time': [df_raw.index.min()],
        'end_time': [df_raw.index.max()],
    })
    save_table(info_table, 'basic_info', metric_dir)

    base_cols = [c for c in ['open', 'close', 'high', 'low', 'volume', 'money'] if c in df_raw.columns]
    desc_table = basic_describe(df_raw, base_cols)
    save_table(desc_table, 'price_volume_describe', metric_dir)

    df_feat = add_return_columns(df_raw)
    ret_stats = return_stats(df_feat['r'].dropna())
    save_table(ret_stats, 'return_stats', metric_dir)

    plot_close_volume(df_raw, fig_dir, short_tag, resample_rule=RESAMPLE_RULE)
    plot_price_volume_scatter(df_raw, fig_dir, short_tag, sample_points=SAMPLE_POINTS)
    plot_return_timeseries(df_feat, fig_dir, short_tag)
    plot_return_histograms(df_feat, fig_dir, short_tag)
    plot_volatility_proxy(df_feat, fig_dir, short_tag)
    plot_acf_panels(df_feat, fig_dir, short_tag, lags=ACF_LAGS)
    plot_intraday_patterns(df_feat, fig_dir, short_tag)
    plot_volatility_vs_volume(df_feat, fig_dir, short_tag, sample_points=SAMPLE_POINTS)
    plot_correlation_heatmap(df_feat, fig_dir, short_tag)

    return {
        'short_tag': short_tag,
        'data': df_feat,
        'fig_dir': fig_dir,
        'metric_dir': metric_dir,
    }


## 1. Data loading and basic info

This section loads one or more main-contract CSV files, applies optional date filters, and reports basic coverage (row count and time span).

## 2. Price and volume basics

We visualize close prices and trading volume over time (downsampled) and inspect the price–volume relationship with scatter plots.

## 3. Return construction and distribution

We compute minute log returns (in percentage points) and study their temporal evolution and distribution against a Gaussian benchmark.

## 4. Volatility proxies and autocorrelation

We examine absolute and squared returns, price ranges, and their autocorrelations to highlight volatility clustering.

## 5. Intraday patterns

We explore time-of-day effects in volatility (|r|) and trading volume, typically yielding the well-known U-shape across the trading day.

## 6. Relationships among variables

We relate volatility proxies to trading activity via scatter plots and correlation heatmaps.

In [ ]:

# -----------------------------
# Run notebook logic
# -----------------------------
np.random.seed(RANDOM_SEED)

csv_files = list_csv_files(DATA_DIR, limit=MAX_FILES)
contract_results: Dict[str, Dict] = {}

for path in csv_files:
    res = run_descriptive_pipeline(path)
    if res:
        contract_results[res['short_tag']] = res

if len(contract_results) >= 2:
    multi_contract_volatility_summary(contract_results, METRIC_ROOT, FIG_ROOT)
else:
    print('Multi-contract comparison skipped (need at least 2 contracts).')
